# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Lane:** content-refresh triage. The question this baseline answers: *which pages should a
human look at first for a refresh?* A page is worth that attention if it (a) hasn't been touched
in a long time and (b) is still pulling in real search visibility — meaning there's something
there worth protecting before it decays further.

### Signal check 1 — staleness (behind the refresh flags)

Signal: `days_since_last_update`, read through `freshness_tier`.
Claim being tested: pages that have gone longer without an update are more likely to already be
declining (`is_declining_label`). `is_declining_label` is used here **only to check the signal**
— it never enters the score in Section 2.
Bucket table: one row per `freshness_tier`, with `n` and the observed decline rate.
Verdict is printed under the table by the code below (CONFIRMED / OPPOSITE / MIXED / FALSE).

### Signal check 2 — CTR vs. position (behind the CTR-fix logic)

Signal: `ctr`, read through `position_tier` (the tier already encodes "vs. position").
Claim being tested: CTR should climb as position improves — `top_3` > `page_1` > `striking` >
`page_3_5` > `deep`. A flat or inverted pattern would mean a "good position, weak CTR" flag is
chasing noise rather than a real pattern.
Caution from the data dictionary: `top_3` in this slice has a thin median volume, where a
single click swings CTR by multiple points — the verdict below is read next to `n`, not instead
of it.

### The rule, in plain words

"A page is worth flagging for refresh if it hasn't been updated in 91+ days **and** it's still
pulling in real search visibility (≥500 impressions in the trailing 90 days). Among pages that
clear both bars, the ones with more current impression volume are worth reviewing first —
that's where the most visibility is at stake."

### Score, reason code, action label

- **Score:** `stale * visible * impressions_90d` (readable on purpose — no fitted weights)
- **Reason code (one, applied to every scored row):** `stale_but_visible`
- **Action label:** `review_for_refresh`

No future-window or label-derived columns (`trend_pct`, `trend_direction`,
`is_declining_label`) are used in the score itself — only in the signal-check tables above and
the precision@K evaluation in Section 2, both of which are checks on the rule, not inputs to it.


In [ ]:
import pandas as pd

pd.set_option("display.width", 120)

# --- load the starter dataset ---
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"loaded {len(df):,} rows x {df.shape[1]} columns")

# ============================================================
# Signal check 1: staleness -> decline rate, by freshness_tier
# ============================================================
freshness_order = ["0-30", "31-90", "91-180", "181+"]  # 'never' has 0 rows in this slice

sig1 = (
    df[df["freshness_tier"].isin(freshness_order)]
    .assign(freshness_tier=lambda d: pd.Categorical(
        d["freshness_tier"], categories=freshness_order, ordered=True
    ))
    .groupby("freshness_tier", observed=True)["is_declining_label"]
    .agg(n="size", decline_rate="mean")
    .reindex(freshness_order)
)
print("\nSignal 1 - staleness vs decline rate (freshness_tier)")
print(sig1)

rates_1 = sig1["decline_rate"].values
base_rate = df["is_declining_label"].mean()
print(f"\noverall base rate (all rows): {base_rate:.3f}")

# verdict: does decline rate rise as freshness_tier gets staler?
is_monotonic_up = all(rates_1[i] <= rates_1[i + 1] + 1e-9 for i in range(len(rates_1) - 1))
spread = rates_1.max() - rates_1.min()
if is_monotonic_up and spread >= 0.03:
    verdict_1 = "CONFIRMED"
elif spread < 0.01:
    verdict_1 = "FALSE"
elif rates_1[-1] < rates_1[0]:
    verdict_1 = "OPPOSITE"
else:
    verdict_1 = "MIXED"
print(f"VERDICT (signal 1 - staleness): {verdict_1}  (spread={spread:.3f})")

# ============================================================
# Signal check 2: CTR vs position, by position_tier
# ============================================================
position_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]  # best -> worst, no_data dropped

sig2 = (
    df[df["position_tier"].isin(position_order)]
    .assign(position_tier=lambda d: pd.Categorical(
        d["position_tier"], categories=position_order, ordered=True
    ))
    .groupby("position_tier", observed=True)["ctr"]
    .agg(n="size", mean_ctr="mean", median_ctr="median")
    .reindex(position_order)
)
print("\nSignal 2 - CTR vs position (position_tier)")
print(sig2)

ctr_vals = sig2["mean_ctr"].values
is_monotonic_down = all(ctr_vals[i] >= ctr_vals[i + 1] - 1e-9 for i in range(len(ctr_vals) - 1))
ctr_spread = ctr_vals.max() - ctr_vals.min()
if is_monotonic_down and ctr_spread >= 0.1:
    verdict_2 = "CONFIRMED"
elif ctr_spread < 0.02:
    verdict_2 = "FALSE"
elif ctr_vals[-1] > ctr_vals[0]:
    verdict_2 = "OPPOSITE"
else:
    verdict_2 = "MIXED"
print(f"VERDICT (signal 2 - CTR vs position): {verdict_2}  (spread={ctr_spread:.3f}pp)")
print(
    "note: top_3 here runs on thin volume (see data dictionary) - read this verdict "
    "next to n, not instead of it."
)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os
import numpy as np

# --- transparent rule: stale AND visible, scaled by current visibility ---
STALE_DAYS_THRESHOLD = 180          # "91+" bucket boundary in freshness_tier is 91-180 / 181+;
                                     # using the harder 180-day cut keeps the flag conservative
VISIBLE_IMPRESSIONS_THRESHOLD = 500

stale = (df["days_since_last_update"] >= STALE_DAYS_THRESHOLD).astype(int)
visible = (df["impressions_90d"] >= VISIBLE_IMPRESSIONS_THRESHOLD).astype(int)

df["score"] = stale * visible * df["impressions_90d"]          # readable on purpose
df["reason_code"] = "stale_but_visible"
df["action_label"] = np.where(df["score"] > 0, "review_for_refresh", "no_action")

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

n_flagged = (ranked["score"] > 0).sum()
print(f"{n_flagged:,} of {len(ranked):,} rows flagged (score > 0)")

# --- write the ranked queue (not committed to git - regenerated every run) ---
os.makedirs("work/outputs", exist_ok=True)
out_cols = [
    "content_id", "client_id", "score", "reason_code", "action_label",
    "days_since_last_update", "impressions_90d", "ctr", "avg_position",
]
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)
print("wrote work/outputs/baseline_action_score.csv")

# --- honest evaluation: precision@K against the label, next to the base rate ---
def precision_at_k(labels, k):
    return labels.iloc[:k].mean()

base_rate = df["is_declining_label"].mean()
print(f"\nbase rate (is_declining_label over all rows): {base_rate:.3f}")
for k in (20, 50, 200):
    p_at_k = precision_at_k(ranked["is_declining_label"], k)
    print(f"precision@{k}: {p_at_k:.3f}  (vs base rate {base_rate:.3f})")


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = ranked.head(20).copy()

def confidence_note(row):
    # simple, data-driven confidence read: bigger margin above both thresholds = higher confidence
    stale_margin = row["days_since_last_update"] - STALE_DAYS_THRESHOLD
    visible_margin = row["impressions_90d"] - VISIBLE_IMPRESSIONS_THRESHOLD
    if stale_margin > 90 and visible_margin > 2000:
        return "high - well past both thresholds"
    if stale_margin > 30 and visible_margin > 500:
        return "medium - clears both thresholds with room"
    return "low - close to one of the thresholds, worth a second look"

def would_be_wrong_if(row):
    notes = []
    if row["content_type"] == "feedly article":
        notes.append("this content_type often has no keyword data - confirm it's not aggregator content that isn't meant to be 'fresh'")
    if pd.notna(row.get("search_volume")) and row.get("search_volume", 0) < 50:
        notes.append("target keyword volume is low, so the impression count may be a temporary spike rather than durable visibility")
    if row["avg_position"] == 0:
        notes.append("avg_position = 0 means no position data - impressions may be coming from a source GSC isn't attributing cleanly")
    if not notes:
        notes.append("its next 30-day impressions actually hold or grow instead of slipping - would undercut the 'worth protecting' premise")
    return "; ".join(notes)

print(f"{'#':<3}{'content_id':<22}{'action':<20}{'reason_code':<20}{'confidence'}")
for i, row in top20.iterrows():
    print(f"{i+1:<3}{row['content_id']:<22}{row['action_label']:<20}{row['reason_code']:<20}{confidence_note(row)}")
    print(f"     why: {row['days_since_last_update']:.0f} days stale, {row['impressions_90d']:.0f} impressions/90d, score={row['score']:.0f}")
    print(f"     would be wrong if: {would_be_wrong_if(row)}")
    print()


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# --- weak picks: rows in the top 20 that are borderline on one of the two conditions ---
top20["stale_margin"] = top20["days_since_last_update"] - STALE_DAYS_THRESHOLD
top20["visible_margin"] = top20["impressions_90d"] - VISIBLE_IMPRESSIONS_THRESHOLD

weak = top20[(top20["stale_margin"] < 15) | (top20["visible_margin"] < 100)]
print(f"{len(weak)} of the top 20 are borderline (within a thin margin of a threshold):")
if len(weak):
    print(weak[["content_id", "days_since_last_update", "impressions_90d", "stale_margin", "visible_margin"]])
else:
    print("none found in this run - every top-20 row clears both thresholds with real room. "
          "Worth lowering VISIBLE_IMPRESSIONS_THRESHOLD or STALE_DAYS_THRESHOLD in a follow-up "
          "pass to force a genuinely weak pick into view, per the skill's 'if it found none, "
          "look harder' check.")

# --- leakage check: confirm the score never touched a label-derived or future-window column ---
banned_columns = {"trend_pct", "trend_direction", "is_declining_label"}
score_inputs = {"days_since_last_update", "impressions_90d"}
leak = banned_columns & score_inputs
print(f"\nscore built from: {sorted(score_inputs)}")
print(f"banned (label-derived) columns: {sorted(banned_columns)}")
print(f"leakage check: {'FAIL - leaked columns used' if leak else 'PASS - no overlap'}")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.